## Configuring Thinking with the Claude API

### Installing Utilities and libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=claude_api_key)

### Setting the User Prompt

In [ ]:
user_prompt = """
You are a senior digital marketing consultant.

A client's e-commerce website has experienced the following issues over the past three months:

- Website traffic has decreased by 35%
- Conversion rate has dropped from 3.8% to 1.7%
- Bounce rate has increased to 72%
- Mobile visitors account for 80% of all traffic

Recommend a recovery strategy.

For each recommendation:

- Explain why it should be prioritized.
- Describe its expected business impact.
- Identify any risks or trade-offs.
"""

### Configuring Thinking with the Claude API

In [ ]:
response = client.messages.create(
    model = claude_model_name,
    max_tokens = 20000,
    thinking = {"type": "adaptive", "display": "summarized"},
    messages = [
        {
            "role": "user",
            "content": user_prompt
        }
    ]
)

for block in response.content:
    if block.type == "thinking":
        print(f"\nThinking: {block.thinking}")
    elif block.type == "text":
        print(f"\nResponse: {block.text}")

### Stream Output with Adaptive Thinking

In [ ]:
with client.messages.stream(
    model = claude_model_name,
    max_tokens = 20000,
    thinking = {"type": "adaptive", "display": "summarized"},
    messages = [
        {
            "role": "user",
            "content": user_prompt
        }
    ] 
) as stream:
    for event in stream:
        if event.type == "content_block_start":
            print(f"\nStarting {event.content_block.type} block...")
        elif event.type == "content_block_delta":
            if event.delta.type == "thinking_delta":
                print(event.delta.thinking, end="", flush=True)
            elif event.delta.type == "text_delta":
                print(event.delta.text, end="", flush=True)